In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import time

# ======================================================
# CONFIGURATION
# ======================================================
SAVE_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL"
os.makedirs(SAVE_DIR, exist_ok=True)

TARGET_FRAMES_PER_SIGN = 1000       # 🔥 Collect EXACTLY 1000 clean frames per sign
MAX_FRAMES_PER_SESSION = 80         # 🔥 80 frames per recording session
MIN_MOVEMENT_THRESHOLD = 0.003      # 🔥 Skip frames with too little movement
SKIP_EMPTY_FRAME = True             # 🔥 Avoid glitching & missing hands

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# 🔥 Enhanced Stability Settings
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=2,
    smooth_landmarks=True,
    refine_face_landmarks=False,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
)


# ======================================================
# Extract BOTH HANDS only (126 features)
# ======================================================
def extract_keypoints(results):
    lh = np.zeros(63)
    rh = np.zeros(63)

    if results.left_hand_landmarks:
        lh = np.array([[lm.x, lm.y, lm.z]
                       for lm in results.left_hand_landmarks.landmark]).flatten()

    if results.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z]
                       for lm in results.right_hand_landmarks.landmark]).flatten()

    return np.concatenate([lh, rh])


# ======================================================
# FRAME CLEANER — removes jitter, duplicates, glitches
# ======================================================
def is_valid_frame(prev_frame, current_frame):
    if prev_frame is None:
        return True

    diff = np.mean(np.abs(prev_frame - current_frame))

    # Skip no-movement / duplicate frames
    if diff < MIN_MOVEMENT_THRESHOLD:
        return False

    return True


# ======================================================
# MAIN COLLECTION LOOP
# ======================================================
def record_sign(label):
    label_dir = os.path.join(SAVE_DIR, label)
    os.makedirs(label_dir, exist_ok=True)

    csv_path = os.path.join(label_dir, f"{label}.csv")

    print(f"\n🎬 Recording sign: {label}")
    time.sleep(1)

    # Load previous data if exists
    if os.path.isfile(csv_path):
        df_existing = pd.read_csv(csv_path)
        total_existing_frames = df_existing.shape[0]
        print(f"📦 Existing frames: {total_existing_frames}")
    else:
        df_existing = pd.DataFrame()
        total_existing_frames = 0

    cap = cv2.VideoCapture(0)
    prev_frame = None

    while total_existing_frames < TARGET_FRAMES_PER_SIGN:
        frames_this_session = []
        remaining_needed = TARGET_FRAMES_PER_SIGN - total_existing_frames
        session_goal = min(MAX_FRAMES_PER_SESSION, remaining_needed)

        print(f"\n➡ New session started (Goal this session: {session_goal} frames)")
        print("👉 Press ESC anytime to stop")

        collected = 0

        while collected < session_goal:
            ret, frame = cap.read()
            if not ret:
                continue

            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb)

            hands_detected = results.left_hand_landmarks or results.right_hand_landmarks

            # Skip frames with no hands
            if not hands_detected and SKIP_EMPTY_FRAME:
                cv2.putText(frame, "NO HANDS DETECTED", (10, 160),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                cv2.imshow("Recorder", frame)
                if cv2.waitKey(1) & 0xFF == 27:
                    return
                continue

            keypoints = extract_keypoints(results)

            # Skip bad/jitter frames
            if not is_valid_frame(prev_frame, keypoints):
                continue

            frames_this_session.append(keypoints)
            prev_frame = keypoints.copy()
            collected += 1
            total_existing_frames += 1

            # Draw hand landmarks
            if results.left_hand_landmarks:
                mp_drawing.draw_landmarks(frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

            # UI text
            cv2.putText(frame, f"Label: {label}", (10, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
            cv2.putText(frame, f"Collected: {total_existing_frames}/{TARGET_FRAMES_PER_SIGN}", (10, 75),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

            cv2.imshow("Recorder", frame)

            if cv2.waitKey(1) & 0xFF == 27:
                cap.release()
                cv2.destroyAllWindows()
                return

        # Save session
        session_df = pd.DataFrame(frames_this_session)

        if df_existing.shape[0] == 0:
            session_df.to_csv(csv_path, index=False)
        else:
            df_existing = pd.concat([df_existing, session_df], ignore_index=True)
            df_existing.to_csv(csv_path, index=False)

        print(f"✔ Session saved ({collected} frames)")
        print(f"📦 Total so far: {total_existing_frames}/{TARGET_FRAMES_PER_SIGN}")

    cap.release()
    cv2.destroyAllWindows()
    print(f"\n🎉 COMPLETED! Final total: {total_existing_frames} frames for {label}")


# ======================================================
# USER INTERFACE
# ======================================================
if __name__ == "__main__":
    print("\n=== Enhanced Dynamic Dataset Collector ===")
    print("Collecting 1000 CLEAN frames per sign.\n")

    while True:
        label = input("Label: ").strip().upper()
        if not label:
            print("Invalid label.")
            continue
        record_sign(label)



=== Enhanced Dynamic Dataset Collector ===


🎬 Recording sign: CAR

➡ New session started (Goal this session: 80 frames)
👉 Press ESC anytime to stop
✔ Session saved (80 frames)
📦 Total so far: 80/1000

➡ New session started (Goal this session: 80 frames)
👉 Press ESC anytime to stop
✔ Session saved (80 frames)
📦 Total so far: 160/1000

➡ New session started (Goal this session: 80 frames)
👉 Press ESC anytime to stop
✔ Session saved (80 frames)
📦 Total so far: 240/1000

➡ New session started (Goal this session: 80 frames)
👉 Press ESC anytime to stop
✔ Session saved (80 frames)
📦 Total so far: 320/1000

➡ New session started (Goal this session: 80 frames)
👉 Press ESC anytime to stop
✔ Session saved (80 frames)
📦 Total so far: 400/1000

➡ New session started (Goal this session: 80 frames)
👉 Press ESC anytime to stop
✔ Session saved (80 frames)
📦 Total so far: 480/1000

➡ New session started (Goal this session: 80 frames)
👉 Press ESC anytime to stop
✔ Session saved (80 frames)
📦 Total so fa

In [44]:
import os
import pandas as pd
import numpy as np

DATASET_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL"
SEQ_LENGTH = 40   # 40 frames per sequence
STEP = 1          # Overlap of 39 frames (BEST)

for label in os.listdir(DATASET_DIR):
    folder = os.path.join(DATASET_DIR, label)
    if not os.path.isdir(folder):
        continue

    print(f"\nProcessing label: {label}")

    # Find the long CSV file
    csv_files = [f for f in os.listdir(folder) if f.endswith(".csv")]
    if len(csv_files) == 0:
        print("⚠ No CSV. Skipping.")
        continue

    df = pd.read_csv(os.path.join(folder, csv_files[0]))
    data = df.values
    total_frames = len(data)

    # ⭐ Overlapping calculation
    num_sequences = (total_frames - SEQ_LENGTH) // STEP + 1
    num_sequences = max(num_sequences, 0)

    print(f"• Frames available: {total_frames}")
    print(f"• Overlapping sequences created: {num_sequences}")

    seq_dir = os.path.join(folder, "seq_overlapping")
    os.makedirs(seq_dir, exist_ok=True)

    # Create overlapping sequences
    for i in range(num_sequences):
        start = i * STEP
        end = start + SEQ_LENGTH

        seq = data[start:end]

        path = os.path.join(seq_dir, f"{label}_{i+1:04d}.csv")
        pd.DataFrame(seq).to_csv(path, index=False)

    print(f"✔ Saved {num_sequences} overlapping sequences to {seq_dir}")



Processing label: BOOK
• Frames available: 40
• Overlapping sequences created: 1
✔ Saved 1 overlapping sequences to C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL\BOOK\seq_overlapping

Processing label: CAR
• Frames available: 40
• Overlapping sequences created: 1
✔ Saved 1 overlapping sequences to C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL\CAR\seq_overlapping

Processing label: FAMILY
• Frames available: 40
• Overlapping sequences created: 1
✔ Saved 1 overlapping sequences to C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL\FAMILY\seq_overlapping

Processing label: HAPPY
• Frames available: 40
• Overlapping sequences created: 1
✔ Saved 1 overlapping sequences to C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL\HAPPY\seq_overlapping

Processing label: HELLO
• Frames available: 40
• Overlapping sequences created:

In [42]:
# STEP 2 — CHECK HOW MANY SEQUENCES EACH LABEL HAS

import os

DATASET_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL"

for label in os.listdir(DATASET_DIR):
    seq_folder = os.path.join(DATASET_DIR, label, "seq")
    if not os.path.isdir(seq_folder):
        continue

    files = [f for f in os.listdir(seq_folder) if f.endswith(".csv")]
    print(label, "→", len(files), "sequences")


BOOK → 1 sequences
CAR → 1 sequences
FAMILY → 1 sequences
HAPPY → 1 sequences
HELLO → 1 sequences
HOUSE → 1 sequences
HOW → 1 sequences
I_LOVE_YOU → 1 sequences
NO → 1 sequences
PHONE → 1 sequences
READ → 1 sequences
SLEEP → 1 sequences
THANKYOU → 1 sequences
TIRED → 1 sequences
WRITE → 1 sequences
YES → 1 sequences


In [25]:
import os
import numpy as np
import pandas as pd

DATASET_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL"
SAVE_DIR     = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\processed\ASL"
os.makedirs(SAVE_DIR, exist_ok=True)

SEQ_LENGTH = 40
FEATURES = 126  # 63 left, 63 right

def normalize_frame(frame):
    # reshape 126 -> (42, 3) for easier processing
    pts = frame.reshape(-1, 3)

    # remove Z scale differences
    pts[:, 2] /= np.max(np.abs(pts[:, 2])) + 1e-6

    # center around wrist (index 0)
    wrist = pts[0]
    pts -= wrist

    # scale normalization (all hands same size)
    max_val = np.max(np.abs(pts))
    if max_val > 0:
        pts /= max_val

    return pts.reshape(-1)

X, y = [], []
labels = sorted(os.listdir(DATASET_DIR))
label_map = {label: i for i, label in enumerate(labels)}

print("LABEL MAP:", label_map)

for label in labels:
    seq_folder = os.path.join(DATASET_DIR, label, "seq")
    if not os.path.isdir(seq_folder):
        continue

    for file in os.listdir(seq_folder):
        if not file.endswith(".csv"):
            continue

        df = pd.read_csv(os.path.join(seq_folder, file))
        seq = df.values

        # Normalize every frame
        norm_seq = np.array([normalize_frame(f) for f in seq])

        X.append(norm_seq)
        y.append(label_map[label])

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_DIR, "X.npy"), X)
np.save(os.path.join(SAVE_DIR, "y.npy"), y)
np.save(os.path.join(SAVE_DIR, "labels.npy"), labels)

print("✔ Saved normalized dataset!")
print("X shape:", X.shape)
print("y shape:", y.shape)


LABEL MAP: {'BOOK': 0, 'CAR': 1, 'FAMILY': 2, 'HAPPY': 3, 'HELLO': 4, 'HOUSE': 5, 'HOW': 6, 'I_LOVE_YOU': 7, 'NO': 8, 'PHONE': 9, 'READ': 10, 'SLEEP': 11, 'THANKYOU': 12, 'TIRED': 13, 'WRITE': 14, 'YES': 15}
✔ Saved normalized dataset!
X shape: (16, 40, 126)
y shape: (16,)


In [29]:
import os

DATASET_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL"

for label in os.listdir(DATASET_DIR):
    seq_folder = os.path.join(DATASET_DIR, label, "seq")
    if not os.path.isdir(seq_folder):
        continue

    files = [f for f in os.listdir(seq_folder) if f.endswith(".csv")]
    print(label, "→", len(files), "sequences")


BOOK → 1 sequences
CAR → 1 sequences
FAMILY → 1 sequences
HAPPY → 1 sequences
HELLO → 1 sequences
HOUSE → 1 sequences
HOW → 1 sequences
I_LOVE_YOU → 1 sequences
NO → 1 sequences
PHONE → 1 sequences
READ → 1 sequences
SLEEP → 1 sequences
THANKYOU → 1 sequences
TIRED → 1 sequences
WRITE → 1 sequences
YES → 1 sequences


In [30]:
import numpy as np
from sklearn.model_selection import train_test_split

DATASET_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\processed\ASL"

X = np.load(f"{DATASET_DIR}/X.npy")
y = np.load(f"{DATASET_DIR}/y.npy")

print("Before filtering:", X.shape, y.shape)

# ==========================================================
# REMOVE CLASSES WITH < 2 SAMPLES
# ==========================================================
valid_indices = []
for label in np.unique(y):
    idxs = np.where(y == label)[0]
    if len(idxs) >= 2:
        valid_indices.extend(idxs)
    else:
        print(f"⚠ Removing label {label} (only 1 sample)")

X = X[valid_indices]
y = y[valid_indices]

print("After filtering:", X.shape, y.shape)

# ==========================================================
# SAFE TRAIN/VAL SPLIT
# ==========================================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

np.save(f"{DATASET_DIR}/X_train.npy", X_train)
np.save(f"{DATASET_DIR}/X_val.npy", X_val)
np.save(f"{DATASET_DIR}/y_train.npy", y_train)
np.save(f"{DATASET_DIR}/y_val.npy", y_val)

print("Train:", X_train.shape)
print("Val:", X_val.shape)


Before filtering: (16, 40, 126) (16,)
⚠ Removing label 0 (only 1 sample)
⚠ Removing label 1 (only 1 sample)
⚠ Removing label 2 (only 1 sample)
⚠ Removing label 3 (only 1 sample)
⚠ Removing label 4 (only 1 sample)
⚠ Removing label 5 (only 1 sample)
⚠ Removing label 6 (only 1 sample)
⚠ Removing label 7 (only 1 sample)
⚠ Removing label 8 (only 1 sample)
⚠ Removing label 9 (only 1 sample)
⚠ Removing label 10 (only 1 sample)
⚠ Removing label 11 (only 1 sample)
⚠ Removing label 12 (only 1 sample)
⚠ Removing label 13 (only 1 sample)
⚠ Removing label 14 (only 1 sample)
⚠ Removing label 15 (only 1 sample)
After filtering: (0, 40, 126) (0,)


ValueError: With n_samples=0, test_size=0.15 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [1]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.utils import to_categorical

DATASET_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\dataset\ASL"
SAVE_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\processed\ASL"
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_FRAMES = 40               # Fixed sequence length
FEATURES = 126                # 21 keypoints × 2 hands × 3 coords

X, y = [], []
labels = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d)) and 
       len(os.listdir(os.path.join(DATASET_DIR, d))) > 0
])

label_map = {label: i for i, label in enumerate(labels)}

print("Found labels:", labels)

def fix_length(seq):
    """Pad or trim recording to MAX_FRAMES."""
    if len(seq) > MAX_FRAMES:
        return seq[:MAX_FRAMES]
    pad = np.zeros((MAX_FRAMES - seq.shape[0], FEATURES))
    return np.vstack([seq, pad])

for label in labels:
    label_folder = os.path.join(DATASET_DIR, label)

    for file in os.listdir(label_folder):
        if not file.endswith(".csv"):
            continue

        df = pd.read_csv(os.path.join(label_folder, file))

        # 🔥 FIX: Support CSVs with or without frame_index
        if "frame_index" in df.columns:
            df = df.drop(columns=["frame_index"])

        seq = df.values  # (frames, 126)

        seq = fix_length(seq)
        X.append(seq)
        y.append(label_map[label])

X = np.array(X)                           # (samples, 40, 126)
y = to_categorical(y, num_classes=len(labels))

print("X shape:", X.shape)
print("y shape:", y.shape)

np.save(os.path.join(SAVE_DIR, "X.npy"), X)
np.save(os.path.join(SAVE_DIR, "y.npy"), y)
np.save(os.path.join(SAVE_DIR, "labels.npy"), labels)

print("\n✅ Dataset processed successfully!")


Found labels: ['BOOK', 'CAR', 'FAMILY', 'HAPPY', 'HELLO', 'HOUSE', 'HOW', 'I_LOVE_YOU', 'NO', 'PHONE', 'READ', 'SLEEP', 'THANKYOU', 'TIRED', 'WRITE', 'YES']
X shape: (16, 40, 126)
y shape: (16, 16)

✅ Dataset processed successfully!


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import os

DATA_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\processed\ASL"
MODEL_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\models\ASL"
os.makedirs(MODEL_DIR, exist_ok=True)

X = np.load(f"{DATA_DIR}/X.npy")
y = np.load(f"{DATA_DIR}/y.npy")
labels = np.load(f"{DATA_DIR}/labels.npy", allow_pickle=True)

SEQUENCE_LEN = X.shape[1]     # 40
FEATURES = X.shape[2]         # 126
NUM_CLASSES = y.shape[1]

print("Training data:", X.shape, y.shape)

model = models.Sequential([
    layers.Masking(mask_value=0., input_shape=(SEQUENCE_LEN, FEATURES)),
    layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X, y,
    epochs=50,
    batch_size=16,
    validation_split=0.2
)

model_path = os.path.join(MODEL_DIR, "asl_dynamic_lstm.keras")
model.save(model_path)

print("\n🎉 Training complete!")
print("Saved model →", model_path)


Training data: (16, 40, 126) (16, 16)
Epoch 1/50
1/1 [==============================] - 30s 30s/step - loss: 2.8325 - accuracy: 0.1667 - val_loss: 2.8904 - val_accuracy: 0.0000e+00
Epoch 2/50
1/1 [==============================] - 0s 211ms/step - loss: 2.6979 - accuracy: 0.0833 - val_loss: 3.0703 - val_accuracy: 0.0000e+00
Epoch 3/50
1/1 [==============================] - 0s 207ms/step - loss: 2.6770 - accuracy: 0.0833 - val_loss: 3.2565 - val_accuracy: 0.0000e+00
Epoch 4/50
1/1 [==============================] - 0s 323ms/step - loss: 2.7407 - accuracy: 0.0000e+00 - val_loss: 3.3821 - val_accuracy: 0.0000e+00
Epoch 5/50
1/1 [==============================] - 0s 275ms/step - loss: 2.6402 - accuracy: 0.0000e+00 - val_loss: 3.4828 - val_accuracy: 0.0000e+00
Epoch 6/50
1/1 [==============================] - 0s 208ms/step - loss: 2.5942 - accuracy: 0.1667 - val_loss: 3.5726 - val_accuracy: 0.0000e+00
Epoch 7/50
1/1 [==============================] - 0s 210ms/step - loss: 2.4677 - accuracy: 

In [3]:
import tensorflow as tf
import numpy as np
import os

# Correct paths
MODEL_DIR = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\models\ASL"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "asl_dynamic_lstm.keras")
print("Loading model from:", model_path)

if not os.path.isfile(model_path):
    raise FileNotFoundError(f"❌ Model not found at: {model_path}")

model = tf.keras.models.load_model(model_path)

# ================================
# ⭐ FIX THAT SOLVES LSTM CONVERSION PROBLEM
# ================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# required when using LSTM / Bidirectional
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

# disable TensorList lowering (critical fix)
converter._experimental_lower_tensor_list_ops = False

# enable optimizations (optional)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# ================================
# Convert
# ================================
tflite_model = converter.convert()

# Save final TFLite model
tflite_path = os.path.join(MODEL_DIR, "asl_dynamic_lstm.tflite")

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("\n✔ Exported TFLite model successfully!")
print("Saved →", tflite_path)


Loading model from: C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\models\ASL\asl_dynamic_lstm.keras


INFO:tensorflow:Assets written to: C:\Users\JAMJAY~1\AppData\Local\Temp\tmpgv5w2o01\assets


INFO:tensorflow:Assets written to: C:\Users\JAMJAY~1\AppData\Local\Temp\tmpgv5w2o01\assets



✔ Exported TFLite model successfully!
Saved → C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset\models\ASL\asl_dynamic_lstm.tflite


In [22]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os

# ==========================
# PATHS
# ==========================
BASE = r"C:\Users\JamJayDatuin\Documents\Machine Learning Projects\SignLanguagesDataset"

MODEL_DIR     = os.path.join(BASE, "models", "ASL")
PROCESSED_DIR = os.path.join(BASE, "processed", "ASL")
DATASET_DIR   = os.path.join(BASE, "dataset", "ASL")

MODEL_PATH  = os.path.join(MODEL_DIR, "asl_dynamic_lstm.keras")
LABELS_PATH = os.path.join(PROCESSED_DIR, "labels.npy")

MAX_FRAMES = 40
FEATURES = 126

# Load model + labels
labels = np.load(LABELS_PATH, allow_pickle=True)
model  = tf.keras.models.load_model(MODEL_PATH)

# ==========================
# PICK A SIGN TO TEST
# ==========================
label = "CAR"  # <-- change this
csv_path = os.path.join(DATASET_DIR, label, f"{label}.csv")

df = pd.read_csv(csv_path)

# trim or pad
seq = df.values[:MAX_FRAMES]
if seq.shape[0] < MAX_FRAMES:
    pad = np.zeros((MAX_FRAMES - seq.shape[0], FEATURES))
    seq = np.vstack((seq, pad))

input_data = np.expand_dims(seq, axis=0).astype(np.float32)

# ==========================
# PREDICT
# ==========================
probs = model.predict(input_data, verbose=0)[0]
idx = int(np.argmax(probs))
pred_label = labels[idx]
confidence = float(np.max(probs))

print("\n=== CSV TEST PREDICTION ===")
print("Actual Sign:", label)
print("Predicted Sign:", pred_label)
print("Confidence:", confidence)



=== CSV TEST PREDICTION ===
Actual Sign: CAR
Predicted Sign: I_LOVE_YOU
Confidence: 0.3271253705024719
